# NDCG для англоязычных видео

В этом ноутбуке измеряется качество ранжирования английских видеороликов по текстовому запросу при помощью метрики MAP, NDCG используя:

- модель: `openai/clip-vit-base-patch32`
- датасет: `hltcoe/MultiVENT2.0` (нужен HF-токен)
- векторное хранилище: `Weaviate`

In [ ]:
!pip -q install decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 100.9 MB/s eta 0:00:00


In [ ]:
!pip -q install weaviate-client>=4.16.0

In [ ]:
!pip -q install ir_measures

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 30.1 MB/s eta 0:00:00


In [ ]:
import os
import json
import tarfile
import glob
import re
import numpy as np
import torch
import weaviate
import weaviate.classes as wvc
import ir_measures
import csv as _csv
import time
import statistics

from ir_measures import AP, nDCG
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from decord import VideoReader, cpu
from transformers import CLIPModel, CLIPProcessor
from weaviate.classes.query import MetadataQuery
from google.colab import userdata
from huggingface_hub import login, whoami, hf_hub_download, list_repo_files
from weaviate.classes.config import Configure, Property, DataType, Tokenization
from weaviate.classes.config import VectorDistances
from weaviate.classes.query import HybridFusion

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


# Установка зависимостей.

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

login(token=HF_TOKEN, add_to_git_credential=False)
print("Вход в HF как:", whoami(token=HF_TOKEN).get("name"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Вход в HF как: Andrey32


In [ ]:
DATASET_ID = "hltcoe/MultiVENT2.0"
MODEL_NAME = "openai/clip-vit-base-patch32"

DATA_DIR = Path("/content/multivent2_data")
TRAIN_DIR = DATA_DIR / "train_extracted"
TEST_DIR = DATA_DIR / "test_extracted"
for d in (DATA_DIR, TRAIN_DIR, TEST_DIR):
    d.mkdir(parents=True, exist_ok=True)

NUM_TARS_PER_SPLIT = 80
TARGET_LANGUAGE = "english"
NUM_FRAMES_PER_VIDEO = 4
NDCG_KS = [5, 10, 50, 100]
TEXT_BATCH = 64
FRAME_BATCH = 32

COL_VIDEO_HYB = "MultiVentHybridVideo"
BM25_TOKENIZATION = "lowercase"
HYBRID_ALPHA = 0.5
HYBRID_FUSION = "RANKED"
RRF_K = 60

print("Конфигурация:")
for k, v in dict(
    DATASET_ID=DATASET_ID, MODEL_NAME=MODEL_NAME,
    DATA_DIR=str(DATA_DIR), NUM_TARS_PER_SPLIT=NUM_TARS_PER_SPLIT,
    TARGET_LANGUAGE=TARGET_LANGUAGE, NUM_FRAMES_PER_VIDEO=NUM_FRAMES_PER_VIDEO,
    NDCG_KS=NDCG_KS,
    COL_VIDEO_HYB=COL_VIDEO_HYB, BM25_TOKENIZATION=BM25_TOKENIZATION,
    HYBRID_ALPHA=HYBRID_ALPHA, HYBRID_FUSION=HYBRID_FUSION, RRF_K=RRF_K,
).items():
    print(f"  {k} = {v}")

Конфигурация:
  DATASET_ID = hltcoe/MultiVENT2.0
  MODEL_NAME = openai/clip-vit-base-patch32
  DATA_DIR = /content/multivent2_data
  NUM_TARS_PER_SPLIT = 80
  TARGET_LANGUAGE = english
  NUM_FRAMES_PER_VIDEO = 4
  NDCG_KS = [5, 10, 50, 100]
  COL_VIDEO_HYB = MultiVentHybridVideo
  BM25_TOKENIZATION = lowercase
  HYBRID_ALPHA = 0.5
  HYBRID_FUSION = RANKED
  RRF_K = 60


# Скачивание и фильтрация данных по языку.

In [ ]:
META_FILES = [
    "multivent_2_test_judgments.jsonl",
    "multivent_2_train_judgments.jsonl",
    "multivent_2_test_queries.csv",
    "multivent_2_train_queries.csv",
]

train_tars = []
# train_tars = [f"train/{i:06d}.tar" for i in range(1, NUM_TARS_PER_SPLIT + 1)]
test_tars  = [f"test/{i:06d}.tar"  for i in range(1, NUM_TARS_PER_SPLIT + 1)]

local_paths = {}
for fname in tqdm(META_FILES + train_tars + test_tars, desc="download"):
    local_paths[fname] = hf_hub_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        filename=fname,
        local_dir=str(DATA_DIR),
        token=HF_TOKEN,
    )

print("Скачано файлов:", len(local_paths))

download:   0%|          | 0/84 [00:00<?, ?it/s]

multivent_2_test_judgments.jsonl: 0.00B [00:00, ?B/s]

multivent_2_train_judgments.jsonl: 0.00B [00:00, ?B/s]

multivent_2_test_queries.csv: 0.00B [00:00, ?B/s]

multivent_2_train_queries.csv: 0.00B [00:00, ?B/s]

...

test/000077.tar:   0%|          | 0.00/940M [00:00<?, ?B/s]

test/000078.tar:   0%|          | 0.00/815M [00:00<?, ?B/s]

test/000079.tar:   0%|          | 0.00/970M [00:00<?, ?B/s]

test/000080.tar:   0%|          | 0.00/796M [00:00<?, ?B/s]

Скачано файлов: 84


In [ ]:
def extract_tars(tar_files, out_dir: Path, desc: str):
    for fname in tqdm(tar_files, desc=desc):
        with tarfile.open(local_paths[fname], "r") as tf:
            tf.extractall(out_dir)

extract_tars(train_tars, TRAIN_DIR, "extract train")
extract_tars(test_tars,  TEST_DIR,  "extract test")

stem_to_path = {}
for root in (TRAIN_DIR, TEST_DIR):
    for p in glob.glob(str(root / "**" / "*.mp4"), recursive=True):
        stem_to_path[Path(p).stem] = p

print("Локальные mp4 файлы:", len(stem_to_path))
sample = list(stem_to_path.items())[:5]
print("Путь файлов:", sample)

extract train: 0it [00:00, ?it/s]

extract test:   0%|          | 0/80 [00:00<?, ?it/s]

/tmp/ipykernel_11379/851634606.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(out_dir)


Локальные mp4 файлы: 8000
Путь файлов: [('7722', '/content/multivent2_data/test_extracted/000078/7722.mp4'), ('7781', '/content/multivent2_data/test_extracted/000078/7781.mp4'), ('7702', '/content/multivent2_data/test_extracted/000078/7702.mp4'), ('7718', '/content/multivent2_data/test_extracted/000078/7718.mp4'), ('7703', '/content/multivent2_data/test_extracted/000078/7703.mp4')]


In [ ]:
def load_query_texts(path: str):
    query_texts = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = _csv.DictReader(f)
        for row in reader:
            qid = row.get("query_id") or row.get("Query_id")
            text = row.get("query")
            if qid and text:
                query_texts[str(qid)] = text
    return query_texts


test_query_texts = load_query_texts(local_paths["multivent_2_test_queries.csv"])
train_query_texts = load_query_texts(local_paths["multivent_2_train_queries.csv"])


def infer_video_language(stem: str):
    path = stem_to_path.get(stem)
    if not path:
        return None

    json_p = Path(path).with_suffix(".json")
    if not json_p.exists():
        return None

    try:
        with open(json_p, "r", encoding="utf-8") as f:
            meta = json.load(f)
        return str(meta.get("lang") or "").lower()
    except Exception:
        return None


def is_target_language(obj, stem: str):
    # test judgments have full language names
    lang = str(obj.get("video_language") or "").lower()
    if lang:
        return lang == TARGET_LANGUAGE

    # train judgments do not have video_language; infer from local JSON metadata
    meta_lang = infer_video_language(stem)
    if TARGET_LANGUAGE == "english":
        return meta_lang in {"en", "eng", "english"}

    return meta_lang == TARGET_LANGUAGE


def load_pool(path: str, source_tag: str, query_texts: dict):
    qrels = defaultdict(dict)
    queries_out = {}
    docid_to_stem = {}

    n_total = n_lang = n_local = n_missing_query = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)
            n_total += 1

            doc_id = str(obj["doc_id"])
            orig = str(obj.get("original_doc_id") or "")

            if doc_id in stem_to_path:
                stem = doc_id
            elif orig and orig in stem_to_path:
                stem = orig
            else:
                continue

            if not is_target_language(obj, stem):
                continue

            n_lang += 1

            raw_qid = str(obj["query_id"])
            qtext = obj.get("query") or query_texts.get(raw_qid)
            if not qtext:
                n_missing_query += 1
                continue

            n_local += 1
            qid = f"{source_tag}::{raw_qid}"

            qrels[qid][doc_id] = int(obj["relevance"])
            queries_out[qid] = qtext
            docid_to_stem[doc_id] = stem

    print(
        f"[{source_tag}] Всего: {n_total}  Язык={TARGET_LANGUAGE}: {n_lang}  "
        f"Локально с query: {n_local}  Без query: {n_missing_query}  "
        f"Запросы: {len(qrels)}"
    )

    return qrels, queries_out, docid_to_stem


test_q, test_qt, test_d2s = load_pool(
    local_paths["multivent_2_test_judgments.jsonl"],
    "test",
    test_query_texts,
)

train_q, train_qt, train_d2s = load_pool(
    local_paths["multivent_2_train_judgments.jsonl"],
    "train",
    train_query_texts,
)

qrels = {**test_q, **train_q}
queries = {**test_qt, **train_qt}
docid_to_stem = {**test_d2s, **train_d2s}

usable_qids = [qid for qid, d in qrels.items() if any(r > 0 for r in d.values())]
qrels = {qid: qrels[qid] for qid in usable_qids}
queries = {qid: queries[qid] for qid in usable_qids}

print(f"Запросы, доступные для оценки: {len(qrels)}")

pool_doc_ids = sorted(docid_to_stem.keys())
pool_paths = [stem_to_path[docid_to_stem[d]] for d in pool_doc_ids]

print(f"Доступные английские видео: {len(pool_doc_ids)}")

[test] Всего: 16116  Язык=english: 253  Локально с query: 253  Без query: 0  Запросы: 198
[train] Всего: 1520  Язык=english: 0  Локально с query: 0  Без query: 0  Запросы: 0
Запросы, доступные для оценки: 156
Доступные английские видео: 59


In [ ]:
def load_doc_text(mp4_path: str) -> str:
    p = Path(mp4_path)
    csv_p = p.with_suffix(".csv")
    if csv_p.exists():
        try:
            with open(csv_p, "r", encoding="utf-8") as f:
                reader = _csv.DictReader(f)
                row = next(reader, None)
                if row and "text" in row and row["text"]:
                    return row["text"].strip()
        except Exception as e:
            print("Не удалось прочитать CSV", csv_p, "->", e)
    json_p = p.with_suffix(".json")
    if json_p.exists():
        try:
            with open(json_p, "r", encoding="utf-8") as f:
                meta = json.load(f)
            info = (meta.get("yt_meta_dict") or {}).get("info") or {}
            parts = [info.get("title") or "", info.get("description") or ""]
            parts += list(info.get("tags") or [])
            return " ".join(s for s in parts if s).strip()
        except Exception as e:
            print("Не удалось прочитать JSON", json_p, "->", e)
    return ""

doc_text_by_path = {p: load_doc_text(p) for p in pool_paths}
n_with_text = sum(1 for t in doc_text_by_path.values() if t)
print(f"Документов с текстом: {n_with_text}/{len(pool_paths)}")
sample = next((p for p in pool_paths if doc_text_by_path.get(p)), None)
if sample:
    snippet = doc_text_by_path[sample][:200].replace("\n", " ")
    print(f"Пример [{Path(sample).stem}]: {snippet}...")

Документов с текстом: 59/59
Пример [1048]: Sky's Paul Kelso asked Jeremy Hunt if Brexit is failing for businesses and whether those terms need to be readdressed. In response, the chancellor said: "Brexit is an opportunity and we can make it an...


# Загрузка CLIP, получение текстовых и видео эмбеддингов.

In [ ]:
from transformers import AutoModel, AutoProcessor

MODEL_NAME = "google/siglip-base-patch16-224"

clip_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
clip_processor = AutoProcessor.from_pretrained(MODEL_NAME)
clip_model.eval()
torch.set_grad_enabled(False)

amp_dtype = torch.float16 if device == "cuda" else torch.float32

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def sample_frames_uniform(path: str, num_frames: int) -> np.ndarray:
    vr = VideoReader(path, ctx=cpu(0))
    n = len(vr)
    if n == 0:
        raise ValueError(f"Нет видео: {path}")
    if n <= num_frames:
        idx = np.arange(n)
    else:
        idx = np.linspace(0, n - 1, num_frames).astype(int)
    return vr.get_batch(idx).asnumpy()  # (T, H, W, 3) uint8


def _to_tensor(x):
    if isinstance(x, torch.Tensor):
        return x
    if hasattr(x, "pooler_output"):
        return x.pooler_output
    if hasattr(x, "last_hidden_state"):
        return x.last_hidden_state
    return x[0]


@torch.no_grad()
def _clip_text_features(input_ids, attention_mask):
    return clip_model.get_text_features(input_ids=input_ids, attention_mask=attention_mask)


@torch.no_grad()
def _clip_image_features(pixel_values):
    return clip_model.get_image_features(pixel_values=pixel_values)


@torch.no_grad()
def encode_text_batch(texts):
    if not texts:
        D = getattr(clip_model.config, "text_config", clip_model.config).hidden_size
        return torch.empty(0, D)
    feats = []
    for i in range(0, len(texts), TEXT_BATCH):
        batch = texts[i : i + TEXT_BATCH]
        inputs = clip_processor(
            text=batch, return_tensors="pt", padding=True, truncation=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.autocast(device_type="cuda" if device == "cuda" else "cpu",
                            dtype=amp_dtype, enabled=(device == "cuda")):
            out = _clip_text_features(inputs["input_ids"], inputs["attention_mask"])
        out = out / out.norm(dim=-1, keepdim=True)
        feats.append(out.detach().to(torch.float32).cpu())
    return torch.cat(feats, dim=0)


@torch.no_grad()
def encode_videos(paths):
    feats = {}
    all_frames = []
    counts = []
    valid_paths = []

    for p in tqdm(paths, desc="Обработано видеофайлов"):
        try:
            frames = sample_frames_uniform(p, NUM_FRAMES_PER_VIDEO)
            for t in range(frames.shape[0]):
                all_frames.append(frames[t])
            counts.append(frames.shape[0])
            valid_paths.append(p)
        except Exception as e:
            print("пропуск", p, "->", e)

    if not valid_paths:
        return feats

    chunks = []
    for i in tqdm(range(0, len(all_frames), FRAME_BATCH), desc="Количество батчей для обработки"):
        batch = all_frames[i : i + FRAME_BATCH]
        inputs = clip_processor(images=batch, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)
        with torch.autocast(device_type="cuda" if device == "cuda" else "cpu",
                            dtype=amp_dtype, enabled=(device == "cuda")):
            out = _clip_image_features(pixel_values)
        out = out / out.norm(dim=-1, keepdim=True)
        chunks.append(out.detach().to(torch.float32).cpu())
    frame_feats = torch.cat(chunks, dim=0)

    start = 0
    for path, c in zip(valid_paths, counts):
        v = frame_feats[start : start + c].mean(dim=0)
        v = v / v.norm()
        feats[path] = v
        start += c
    return feats

In [ ]:
# Переопределяем функции для корректного извлечения тензоров из объектов transformers
def _clip_image_features_fixed(pixel_values):
    out = clip_model.get_image_features(pixel_values=pixel_values)
    return _to_tensor(out)

def _clip_text_features_fixed(input_ids, attention_mask=None):
    if attention_mask is not None:
        out = clip_model.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
    else:
        out = clip_model.get_text_features(input_ids=input_ids)
    return _to_tensor(out)

# Исправляем функцию encode_text_batch, чтобы она не требовала attention_mask
@torch.no_grad()
def encode_text_batch_fixed(texts):
    if not texts:
        D = getattr(clip_model.config, "text_config", clip_model.config).hidden_size
        return torch.empty(0, D)
    feats = []
    for i in range(0, len(texts), TEXT_BATCH):
        batch = texts[i : i + TEXT_BATCH]
        inputs = clip_processor(
            text=batch, return_tensors="pt", padding=True, truncation=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.autocast(device_type="cuda" if device == "cuda" else "cpu",
                            dtype=amp_dtype, enabled=(device == "cuda")):
            out = _clip_text_features(inputs["input_ids"], inputs.get("attention_mask"))
        out = out / out.norm(dim=-1, keepdim=True)
        feats.append(out.detach().to(torch.float32).cpu())
    return torch.cat(feats, dim=0)

# Применяем патч в глобальной области видимости
import sys
_main = sys.modules['__main__']
_main._clip_image_features = _clip_image_features_fixed
_main._clip_text_features = _clip_text_features_fixed
_main.encode_text_batch = encode_text_batch_fixed

print("Видео для кодирования:", len(pool_paths))
video_feat_by_path = encode_videos(pool_paths)
print("Закодированных видео:", len(video_feat_by_path))

qids_sorted = sorted(qrels.keys())
qtexts = [queries[qid] for qid in qids_sorted]
text_feats = encode_text_batch(qtexts)
text_feat_by_qid = {qid: text_feats[i] for i, qid in enumerate(qids_sorted)}
print("Закодированные текстовые запросы:", len(text_feat_by_qid))

Видео для кодирования: 59


Обработано видеофайлов:   0%|          | 0/59 [00:00<?, ?it/s]

Количество батчей для обработки:   0%|          | 0/8 [00:00<?, ?it/s]

Закодированных видео: 59
Закодированные текстовые запросы: 156


In [ ]:
ordered_paths = [p for p in pool_paths if p in video_feat_by_path]
doc_texts = [(doc_text_by_path.get(p, "") or " ") for p in ordered_paths]
doc_text_feats = encode_text_batch(doc_texts)
text_feat_by_path = {p: doc_text_feats[i] for i, p in enumerate(ordered_paths)}
print(f"CLIP text-эмбеддинги документов: {len(text_feat_by_path)}")

CLIP text-эмбеддинги документов: 59


# Замер скорости инференса CLIP


In [ ]:
BENCH_BATCH_SIZES = [1, 4, 8, 16, 32, 64]
BENCH_WARMUP = 3
BENCH_ITERS = 10

def _detect_image_hw(processor, default=224):
    img_proc = getattr(processor, "image_processor", processor)
    for attr in ("crop_size", "size"):
        v = getattr(img_proc, attr, None)
        if v is None:
            continue
        if isinstance(v, dict):
            h = v.get("height") or v.get("shortest_edge")
            w = v.get("width")  or v.get("shortest_edge")
            if h and w:
                return int(h), int(w)
        else:
            try:
                s = int(v)
                return s, s
            except Exception:
                pass
    return default, default

_img_h, _img_w = _detect_image_hw(clip_processor)

_proj_dim = getattr(clip_model.config, "projection_dim", 512)
_sample_text = "a man on the news reporting current events about politics in europe"


def _sync():
    if device == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def _bench_step(fn, n_warmup=BENCH_WARMUP, n_iter=BENCH_ITERS):
    for _ in range(n_warmup):
        fn()
    _sync()
    times = []
    for _ in range(n_iter):
        _sync()
        t0 = time.perf_counter()
        fn()
        _sync()
        times.append(time.perf_counter() - t0)
    return times


def _fmt_video_stats(B, times, frames_per_video):
    """
    Переводит время каждой итерации в videos/s и считает статистику.
    """

    vps_list = [B / (t * frames_per_video) for t in times]

    mean_vps = np.mean(vps_list)
    median_vps = np.median(vps_list)
    p95_vps = np.percentile(vps_list, 95)

    return f"{B:>5} | {mean_vps:>12.2f} | {median_vps:>14.2f} | {p95_vps:>11.2f}"


print("=== CLIP image-encoder (videos/s metrics) ===")
header = f"{'Batch':>5} | {'Mean (vid/s)':>12} | {'Median (vid/s)':>14} | {'95p (vid/s)':>11}"
print(header); print("-" * len(header))

img_table = {}
for B in BENCH_BATCH_SIZES:
    pixel_values = torch.randn(B, 3, _img_h, _img_w, device=device)

    def step(pv=pixel_values):
        with torch.autocast(device_type=("cuda" if device == "cuda" else "cpu"),
                            dtype=amp_dtype, enabled=(device == "cuda")):
            out = _clip_image_features(pv)
            out = out / out.norm(dim=-1, keepdim=True)
        return out

    times = _bench_step(step)
    img_table[B] = times
    print(_fmt_video_stats(B, times, NUM_FRAMES_PER_VIDEO))

best_B_img = max(img_table, key=lambda b: np.mean([b / (t * NUM_FRAMES_PER_VIDEO) for t in img_table[b]]))
best_mean_vps = np.mean([best_B_img / (t * NUM_FRAMES_PER_VIDEO) for t in img_table[best_B_img]])

print(f"\nЛучшая пропускная способность: batch={best_B_img}, Среднее: {best_mean_vps:.2f} videos/s")

=== CLIP image-encoder (videos/s metrics) ===
Batch | Mean (vid/s) | Median (vid/s) | 95p (vid/s)
---------------------------------------------------
    1 |        21.79 |          21.98 |       22.41
    4 |        51.47 |          54.69 |       55.49
    8 |        64.99 |          64.90 |       66.15
   16 |        65.59 |          65.60 |       66.22
   32 |        67.23 |          67.49 |       67.86
   64 |        65.87 |          65.95 |       66.18

Лучшая пропускная способность: batch=32, Среднее: 67.23 videos/s


# Загрузка эмбеддингов в Weaviate и подсчёт NDCG

In [ ]:
COL_VIDEO = "MultiVentClipVideo"
COL_QUERY = "MultiVentClipQuery"

def connect_weaviate_client():
    try:
        print("Weaviate: подключение")
        return weaviate.connect_to_embedded()
    except Exception as e:
        print("Weaviate недоступен: пробуем подключится к другому порту")
        return weaviate.connect_to_local(port=8079, grpc_port=50050)

keep_doc_ids, keep_paths = [], []
for i, p in enumerate(pool_paths):
    if p in video_feat_by_path:
        keep_doc_ids.append(pool_doc_ids[i])
        keep_paths.append(p)

if not keep_doc_ids:
    raise RuntimeError("Нет видео с эмбеддингами - нечего индексировать в Weaviate.")

weaviate_client = connect_weaviate_client()

for name in (COL_VIDEO, COL_QUERY):
    if weaviate_client.collections.exists(name):
        weaviate_client.collections.delete(name)

video_coll = weaviate_client.collections.create(
    name=COL_VIDEO,
    vector_index_config=wvc.config.Configure.VectorIndex.hnsw(
        distance_metric=wvc.config.VectorDistances.COSINE,
    ),
    properties=[
        wvc.config.Property(name="doc_id", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="path", data_type=wvc.config.DataType.TEXT),
    ],
)

query_coll = weaviate_client.collections.create(
    name=COL_QUERY,
    vector_index_config=wvc.config.Configure.VectorIndex.hnsw(
        distance_metric=wvc.config.VectorDistances.COSINE,
    ),
    properties=[
        wvc.config.Property(name="qid", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="query_text", data_type=wvc.config.DataType.TEXT),
    ],
)

vid_objs = [
    wvc.data.DataObject(
        properties={"doc_id": doc_id, "path": path},
        vector=video_feat_by_path[path].numpy().tolist(),
    )
    for doc_id, path in zip(keep_doc_ids, keep_paths)
]
video_coll.data.insert_many(vid_objs)

q_objs = [
    wvc.data.DataObject(
        properties={"qid": qid, "query_text": queries[qid]},
        vector=text_feat_by_qid[qid].numpy().tolist(),
    )
    for qid in qids_sorted
]
query_coll.data.insert_many(q_objs)

WEAVIATE_POOL_SIZE = len(keep_doc_ids)
print(
    f"Weaviate: индексировано {WEAVIATE_POOL_SIZE} видео и {len(q_objs)} запросов "
    f"Коллекции '{COL_VIDEO}', '{COL_QUERY}'."
)

INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 44422


Weaviate: подключение
Weaviate: индексировано 59 видео и 156 запросов Коллекции 'MultiVentClipVideo', 'MultiVentClipQuery'.


In [ ]:
_vc = video_coll
_wc = weaviate_client
_n = WEAVIATE_POOL_SIZE

run = {}
for qid in qids_sorted:
    qv = text_feat_by_qid[qid].numpy().tolist()
    res = _vc.query.near_vector(
        near_vector=qv,
        limit=_n,
        return_metadata=MetadataQuery(distance=True, certainty=True),
    )
    rows = {}
    for o in res.objects:
        doc_id = o.properties["doc_id"]
        dmeta = o.metadata
        if dmeta is not None and dmeta.distance is not None:
            s = 1.0 - float(dmeta.distance)
        elif dmeta is not None and dmeta.certainty is not None:
            s = float(dmeta.certainty)
        else:
            s = 0.0
        rows[str(doc_id)] = s
    if len(rows) != _n:
        raise RuntimeError(
            f"Weaviate вернул {len(rows)} объектов для qid={qid}, ожидалось {_n}. "
            "Проверьте limit в near_vector и лимиты кластера."
        )
    run[qid] = rows

measures = (
    [nDCG @ k for k in NDCG_KS]
    + [nDCG]
    + [AP @ k for k in NDCG_KS]
    + [AP]
)
agg = ir_measures.calc_aggregate(measures, qrels, run)

print("\n=== MAP (при помощи библиотеки ir_measures) ===")
for k in NDCG_KS:
    print(f"  AP@{k:<5d} (MAP@K) = {agg[AP @ k]:.4f}")
print(f"  MAP = {agg[AP]:.4f}")

print("\n=== NDCG (при помощи библиотеки ir_measures) ===")
for k in NDCG_KS:
    print(f"  NDCG@{k:<4d} = {agg[nDCG @ k]:.4f}")
print(f"  NDCG       = {agg[nDCG]:.4f}")


=== MAP (при помощи библиотеки ir_measures) ===
  AP@5     (MAP@K) = 0.1441
  AP@10    (MAP@K) = 0.1654
  AP@50    (MAP@K) = 0.1925
  AP@100   (MAP@K) = 0.1946
  MAP = 0.1946

=== NDCG (при помощи библиотеки ir_measures) ===
  NDCG@5    = 0.1732
  NDCG@10   = 0.2206
  NDCG@50   = 0.3430
  NDCG@100  = 0.3600
  NDCG       = 0.3600


# Гибридная коллекция: named vectors `text` + `image` и BM25

Создаём отдельную коллекцию `MultiVentHybridVideo` с двумя именованными векторами (CLIP text-encoder поверх per-video ASR/описания и усреднённый CLIP image-encoder поверх кадров) и инвертированным BM25-индексом по тому же тексту. На этой коллекции дальше посчитаем:

1. **NDCG/MAP только по BM25** — sparse / лексический поиск.
2. **NDCG/MAP по гибридному поиску** с `target_vector=["text","image"]` и `Fusion.RANKED` (RRF).

In [ ]:
if weaviate_client.collections.exists(COL_VIDEO_HYB):
    weaviate_client.collections.delete(COL_VIDEO_HYB)

hyb_coll = weaviate_client.collections.create(
    name=COL_VIDEO_HYB,
    vector_config=[
        Configure.Vectors.self_provided(
            name="text",
            vector_index_config=Configure.VectorIndex.hnsw(
                distance_metric=VectorDistances.COSINE
            ),
        ),
        Configure.Vectors.self_provided(
            name="image",
            vector_index_config=Configure.VectorIndex.hnsw(
                distance_metric=VectorDistances.COSINE
            ),
        ),
    ],
    properties=[
        Property(name="doc_id", data_type=DataType.TEXT),
        Property(name="path", data_type=DataType.TEXT),
        Property(name="text_content", data_type=DataType.TEXT, tokenization=Tokenization.LOWERCASE),
    ],
    inverted_index_config=Configure.inverted_index(bm25_b=0.75, bm25_k1=1.2),
)

hyb_objs = []
for doc_id, path in zip(keep_doc_ids, keep_paths):
    hyb_objs.append(wvc.data.DataObject(
        properties={
            "doc_id": doc_id,
            "path": path,
            "text_content": doc_text_by_path.get(path, "") or "",
        },
        vector={
            "text":  text_feat_by_path[path].numpy().tolist(),
            "image": video_feat_by_path[path].numpy().tolist(),
        },
    ))
hyb_coll.data.insert_many(hyb_objs)
print(
    f"Гибридная коллекция '{COL_VIDEO_HYB}': {len(hyb_objs)} объектов, "
    f"named vectors=['text','image'], BM25 по 'text_content'."
)


Гибридная коллекция 'MultiVentHybridVideo': 59 объектов, named vectors=['text','image'], BM25 по 'text_content'.


In [ ]:
def eval_run(run, label):
    measures = [nDCG @ k for k in NDCG_KS] + [nDCG] + [AP @ k for k in NDCG_KS] + [AP]
    agg = ir_measures.calc_aggregate(measures, qrels, run)
    print(f"\n=== {label} ===")
    for k in NDCG_KS:
        print(f"  AP@{k:<4d} (MAP@K) = {agg[AP @ k]:.4f}")
    print(f"  MAP             = {agg[AP]:.4f}")
    for k in NDCG_KS:
        print(f"  NDCG@{k:<4d}        = {agg[nDCG @ k]:.4f}")
    print(f"  NDCG             = {agg[nDCG]:.4f}")
    return agg

# Метод 1 — BM25 (sparse / лексический поиск)

Чисто лексический baseline через нативный BM25 Weaviate по полю `text_content`. В качестве запроса используем сырой текст `queries[qid]`.

In [ ]:
N = WEAVIATE_POOL_SIZE
run_bm25 = {}
for qid in qids_sorted:
    res = hyb_coll.query.bm25(
        query=queries[qid],
        limit=N,
        return_metadata=MetadataQuery(score=True),
    )
    rows = {}
    for o in res.objects:
        s = float(o.metadata.score) if (o.metadata and o.metadata.score is not None) else 0.0
        rows[str(o.properties["doc_id"])] = s
    run_bm25[qid] = rows

eval_run(run_bm25, "Method 1 — BM25 (sparse only)")


=== Method 1 — BM25 (sparse only) ===
  AP@5    (MAP@K) = 0.7694
  AP@10   (MAP@K) = 0.7728
  AP@50   (MAP@K) = 0.7742
  AP@100  (MAP@K) = 0.7742
  MAP             = 0.7742
  NDCG@5           = 0.7908
  NDCG@10          = 0.7991
  NDCG@50          = 0.8044
  NDCG@100         = 0.8044
  NDCG             = 0.8044


{AP@10: 0.772789479039479,
 AP@5: 0.7694444444444445,
 nDCG@100: 0.8043896357182776,
 AP@50: 0.7741704735239218,
 AP@100: 0.7741704735239218,
 nDCG: 0.8043896357182776,
 nDCG@50: 0.8043896357182776,
 nDCG@5: 0.790835230534843,
 AP: 0.7741704735239218,
 nDCG@10: 0.7990606250163581}

# Метод 2 — Гибридный поиск: named vectors `text` + `image` + BM25, RRF fusion

В одном `query.hybrid` запросе:

- разреженная часть — BM25 по `text_content` (запрос — сырой текст);
- плотная часть — два именованных вектора одновременно (`target_vector=["text", "image"]`); query-вектор берём из CLIP text-encoder, потому что `text` и `image` живут в одном CLIP-проекционном пространстве;
- объединение всех ранжированных списков — `Fusion.RANKED` (Reciprocal Rank Fusion).

In [ ]:
run_hybrid = {}
for qid in qids_sorted:
    qv = text_feat_by_qid[qid].numpy().tolist()
    res = hyb_coll.query.hybrid(
        query=queries[qid],
        vector={"text": qv, "image": qv},
        target_vector=["text", "image"],
        alpha=HYBRID_ALPHA,
        fusion_type=HybridFusion.RANKED,
        limit=N,
        return_metadata=MetadataQuery(score=True),
    )
    rows = {}
    for o in res.objects:
        s = float(o.metadata.score) if (o.metadata and o.metadata.score is not None) else 0.0
        rows[str(o.properties["doc_id"])] = s
    run_hybrid[qid] = rows

eval_run(run_hybrid, "Method 2 — Hybrid (text + image + BM25), Fusion.RANKED")


=== Method 2 — Hybrid (text + image + BM25), Fusion.RANKED ===
  AP@5    (MAP@K) = 0.4619
  AP@10   (MAP@K) = 0.4857
  AP@50   (MAP@K) = 0.4957
  AP@100  (MAP@K) = 0.4964
  MAP             = 0.4964
  NDCG@5           = 0.5133
  NDCG@10          = 0.5648
  NDCG@50          = 0.6066
  NDCG@100         = 0.6107
  NDCG             = 0.6107


{AP@10: 0.4857172534255868,
 AP@5: 0.46194800569800565,
 nDCG@100: 0.6106518717591327,
 AP@50: 0.49573694579040395,
 AP@100: 0.4963983339714022,
 nDCG: 0.6106518717591327,
 nDCG@50: 0.6066456324669138,
 nDCG@5: 0.5132884089202268,
 AP: 0.4963983339714022,
 nDCG@10: 0.5647780833844239}

In [ ]:
try:
    weaviate_client.close()
    print("Weaviate: соединение закрыто.")
except Exception as e:
    print("Weaviate close error:", e)

Weaviate: соединение закрыто.
